# STEP 5 — 평가 · 보정 · 안전장치

여기가 이 프로젝트의 **진짜 목적지**입니다.

목표가 "정확도 높은 모델"이 아니라 **"보호자에게 의심된다고 말해도 되는 시스템"** 이니까요.
그러려면 네 가지가 필요합니다:

| 필요한 것 | 왜 | 이 노트북의 단계 |
|---|---|---|
| 정직한 성능 숫자 | 부풀려진 정확도로 판단하면 안 됨 | 1, 3 |
| 정직한 **확률** | "신뢰도 72%" 가 진짜 72% 여야 함 | 2 |
| 병변을 보고 있다는 증거 | 배경 보고 맞히면 실사용에서 무너짐 | 4 |
| 모르면 모른다고 하기 | 애매한 사진에 답을 지어내면 안 됨 | 5 |

⚠️ 이 노트북의 모든 숫자는 **파이프라인 기준**입니다.
2단계 모델만 따로 잰 macro-F1 은 사용자가 겪는 성능이 아닙니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-25.1"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 0-b. 로컬에서 만든 데이터 불러오기

한국 PC 에서 `prepare_local.py` 로 전처리한 `dogskin_prepared.zip` 을 가져옵니다.

> 🚨 **AI Hub 는 해외 IP 다운로드를 차단**해서 Colab 에서는 원본을 받을 수 없습니다.
> 다운로드·전처리는 로컬에서, 학습만 여기서 합니다.
> → [`docs/cautions/06`](../docs/cautions/06_해외IP_다운로드_차단_우회.md)

**준비**: `dogskin_prepared.zip` 을 Google Drive 에 올려두세요 (Kaggle 이면 Add Input).

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# ★ 인계부터 **먼저** 확인합니다 — 크롭 세는 데만 4분 걸리는데,
#   잘못된 체크포인트를 붙였다면 그 4분이 통째로 낭비입니다.
#   (실제로 8분을 기다린 뒤에야 "크롭이 m1.5 입니다" 를 봤습니다)
from src import train
from src.config import ADOPTED_STAGE2_CROP

# 자동 탐색이 실패하면 여기에 경로를 직접 적으세요 (진단 출력이 후보를 알려줍니다)
#   예: PREV_RUN = "/kaggle/input/dogskin-03-release"
PREV_RUN = None
train.import_previous_run(PREV_RUN)      # 가중치 + stage1_threshold.json 등

_inf = train.infer_run_settings()
if _inf:
    print(f"\n불러온 실행: 1단계 {_inf.get('stage1_crop')} / "
          f"2단계 {_inf.get('stage2_crop')}  ({_inf.get('stage2_exp')})")
    if _inf.get("stage2_crop") and _inf["stage2_crop"] != ADOPTED_STAGE2_CROP:
        raise SystemExit(
            f"\n❌ 2단계 크롭이 '{_inf['stage2_crop']}' 입니다 — 채택된 크롭은 "
            f"'{ADOPTED_STAGE2_CROP}' 입니다 (STEP 4C·4D).\n"
            f"   불러온 체크포인트: {_inf.get('stage2_exp')}\n\n"
            "   → 예전 실행의 출력을 붙였습니다. 03 의 Version 목록에서\n"
            "     m2.5 로 돌린 버전(2시간 18분 / macro-F1 0.5313 / 하락 16.0%)의\n"
            "     release 폴더를 데이터셋으로 만들어 붙이세요.\n"
            "   → 낡은 입력은 Add Input 에서 **빼세요** — 둘 다 있으면 또 걸립니다.")
    print("✅ 채택된 크롭의 체크포인트입니다. 계속 진행합니다.\n")

# Colab: Drive 의 zip 해제 / Kaggle: /kaggle/input 의 풀린 폴더에 링크
env.load_prepared()

_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 끊기면 체크포인트가 사라집니다. env.mount_drive() 확인.")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from src import (labels, split, crop, data, models, train, evaluate,
                 calibrate, explain, infer, stages)
from src.config import (MODEL_BY_KEY, CLASSES_STAGE1, NORMAL_LABEL,
                        ADOPTED_STAGE2_CROP)

# ★ GPU 없이 진행하면 20~30배 느립니다. 없으면 여기서 멈춥니다.
#   (Colab 무료 한도를 넘기면 말없이 CPU 런타임을 줍니다 — 이걸 막습니다)
env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
W = env.work_root()

# 앞 노트북들이 남긴 선택 결과를 읽습니다.
#   03 → stage1_threshold.json  (임계값·크롭·실험 이름)
#   04 → best_model.json        (우승 백본)
# 04 를 건너뛰고 03 → 05 로 바로 와도 동작해야 합니다.
def _load(name):
    p = W/name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}

thr = _load("stage1_threshold.json")        # 노트북 03
crp = _load("reports/step4c_crop.json")     # 노트북 03c (크롭 확정)
sel = _load("best_model.json")              # 노트북 04

# ★ JSON 이 안 넘어와도 체크포인트 **이름**에는 설정이 적혀 있습니다.
#   (stage1_resnet50_full_moderate → 1단계 / resnet50 / full 크롭)
#   가중치는 무거워서 잘 넘어오는데 JSON 은 가벼워서 잘 빠집니다 — 실제로 그랬습니다.
from src import train as _t

inf = _t.infer_run_settings()
if inf and not thr:
    print(f"ℹ️ stage1_threshold.json 이 없어 체크포인트 이름에서 설정을 되살립니다: {inf}")
if not thr and not sel and not inf:
    # ⚠️ 03 을 안 돌린 게 아니라 **인계가 안 된** 경우가 대부분입니다.
    #    Kaggle 은 노트북마다 세션이 따로라 03 의 출력을 입력으로 붙여야 합니다.
    from src import train as _t

    raise FileNotFoundError(
        f"{W} 에 stage1_threshold.json 도 체크포인트도 없습니다.\n"
        "03 을 돌렸다면 그 **출력을 이 노트북 입력으로 붙이지 않은** 것입니다.\n"
        + _t.explain_handoff()
    )

BEST_MODEL  = sel.get("stage2") or inf.get("stage2_model") or (
    "resnet50" if thr else "convnextv2_base")

# 크롭은 03c 가 정합니다 (STEP 4C: m2.5 채택 — 배율 하락 23.2% → 18.8%).
# ⚠️ 03 이 남긴 stage2_crop 은 **일부러 안 봅니다.** 03 은 크롭 비교 이전에 돌아서
#    아직 m1.5 라고 적어두고 있습니다. 낡은 값이 새 결론을 덮어쓰면 안 됩니다.
BEST_CROP   = (sel.get("stage2_crop") or crp.get("best_crop")
               or inf.get("stage2_crop") or "m2.5")
STAGE1_CROP = (sel.get("stage1_crop") or thr.get("stage1_crop")
               or inf.get("stage1_crop") or BEST_CROP)
IMG_SIZE    = int(sel.get("img_size", thr.get("img_size", 384)))

# 1단계가 'full'/'f320' 인 것은 **정상입니다** — ROI 크롭을 쓰면 크롭 창 크기가
# 정답을 흘리는데, 그 신호는 배포에 없습니다 (crop.choose_stage1_tag 참고).
# 촬영 가이드는 2단계 모델로 뽑습니다. 1단계는 배율 정보가 없어 기준이 안 됩니다.
if crop.margin_of_tag(STAGE1_CROP):
    print(f"⚠️ 1단계가 ROI 크롭({STAGE1_CROP}) 입니다 — 지름길을 못 막은 상태입니다.\n"
          f"   1단계 점수를 낙관적으로 취급하세요.")
else:
    print(f"1단계 {STAGE1_CROP} / 2단계 {BEST_CROP} — 의도된 조합입니다 "
          f"(촬영 가이드는 2단계 기준).")

# 배율 → 화면 점유율 변환에 쓰입니다. m1.5 면 1x 에서 병변이 화면의 67%,
# m2.5 면 40% — 즉 크롭이 바뀌면 **보호자에게 할 말이 바뀝니다**.
CROP_MARGIN = crop.margin_of_tag(BEST_CROP) or 1.5

# ★ 붙인 체크포인트가 **채택된 크롭의 것**인지 확인합니다.
#   Kaggle 의 Version 목록에서 예전 실행을 고르기 쉬운데, 그대로 진행하면
#   촬영 가이드·보정·임계값이 전부 버린 설정 기준으로 나오고 에러는 안 납니다.
ALLOW_CROP_MISMATCH = False       # 일부러 다른 크롭을 평가할 때만 True
if BEST_CROP != ADOPTED_STAGE2_CROP and not ALLOW_CROP_MISMATCH:
    raise SystemExit(
        f"❌ 2단계 크롭이 '{BEST_CROP}' 입니다 — 채택된 크롭은 "
        f"'{ADOPTED_STAGE2_CROP}' 입니다 (STEP 4C·4D).\n"
        f"   불러온 체크포인트: {inf.get('stage2_exp') or EXP2 or '(이름 불명)'}\n\n"
        "   → **예전 실행의 출력**을 붙였을 가능성이 큽니다.\n"
        "     Kaggle 노트북의 Version 목록에서 m2.5 로 돌린 버전을 찾아\n"
        "     (2시간 18분 / 2단계 macro-F1 0.5313 / 배율 하락 16.0%)\n"
        "     그 버전의 Output 으로 데이터셋을 **다시** 만들어 붙이세요.\n"
        "   → 일부러 다른 크롭을 보는 거라면 이 셀의 ALLOW_CROP_MISMATCH = True")

# ⚠️ 임계값을 기본값 0.5 로 두면 파이프라인 평가가 조용히 **틀립니다**
#    (03 에서 recall 0.95 로 정한 값은 보통 0.2~0.3 입니다).
#    그래서 못 찾으면 0.5 로 때우지 않고, 아래 6번 셀에서 **다시 계산**합니다.
#    임계값은 "검증셋에서 recall 0.95 가 되는 지점" 이라는 정의뿐이라
#    같은 모델·같은 fold 면 03 과 같은 값이 나옵니다 (holdout 오염 아님).
if "stage1_threshold" in sel:
    THR1 = float(sel["stage1_threshold"])
elif "threshold" in thr:
    THR1 = float(thr["threshold"])
else:
    THR1 = None
    print("⚠️ 1단계 임계값을 못 찾았습니다 → 검증셋에서 다시 계산합니다 (약 1분).")

# 실험 이름은 추측하지 않고 03/04 가 남긴 걸 씁니다
# (with_finetune 이 'stage1_resnet50_full' → 'stage1_resnet50_full_moderate' 로 바꿉니다)
EXP1 = sel.get("stage1_exp") or thr.get("stage1_exp") or inf.get("stage1_exp")
EXP2 = sel.get("stage2_exp") or thr.get("stage2_exp") or inf.get("stage2_exp")
print(f"모델 {BEST_MODEL} | 2단계 크롭 {BEST_CROP}(margin {CROP_MARGIN:g}) | 1단계 크롭 {STAGE1_CROP} | "
      f"입력 {IMG_SIZE}px | 임계값 " + ("(재계산 예정)" if THR1 is None else f"{THR1:.4f}"))
print(f"실험 이름: 1단계 {EXP1 or '(미기록 — 이름으로 탐색)'} / 2단계 {EXP2 or '(미기록)'}")

_raw = labels.load(W/"manifests"/"manifest_final.parquet")
df = crop.switch_tag(_raw, BEST_CROP)
s1_all = stages.to_stage1(crop.switch_tag(_raw, STAGE1_CROP))   # 1단계는 다른 크롭일 수 있음
s2_all = stages.to_stage2(df)

_, va1 = split.get_fold(s1_all, 0)       # 정상 포함
tr2, va2 = split.get_fold(s2_all, 0)     # 병변만
ho1 = split.get_holdout(s1_all)          # 정상 포함 holdout
ho2 = split.get_holdout(s2_all)
print(f"val(전체) {len(va1):,} / val(병변) {len(va2):,} / holdout(전체) {len(ho1):,}")

In [ ]:
# 체크포인트 위치는 어느 노트북에서 왔느냐에 따라 다릅니다.
#   노트북 04 를 돌렸다면  checkpoints/s1_{모델}, s2_{모델}
#   노트북 03 만 돌렸다면  checkpoints/stage1_resnet50_{크롭}, stage2_resnet50_{크롭}
# 03 → 05 로 바로 와도 동작하도록 둘 다 찾습니다.
#
# ★ 학습을 다른 세션에서 했다면 로컬(/content)에는 체크포인트가 없습니다.
#   Drive 백업에서 먼저 되살립니다.
def _find_ckpt(*candidates):
    for name in candidates:
        train.restore_from_persist(name, verbose=False)     # Drive → 로컬
        ck = W/"checkpoints"/name/"best.pt"
        if ck.exists():
            return ck, name
    raise FileNotFoundError(
        "체크포인트를 찾지 못했습니다. 노트북 03(또는 04)을 먼저 돌리세요.\n"
        f"  찾아본 곳: {[str(W/'checkpoints'/n/'best.pt') for n in candidates]}\n"
        f"  로컬에 있는 것: {sorted(p.name for p in (W/'checkpoints').glob('*')) if (W/'checkpoints').exists() else '(폴더 없음)'}\n"
        f"  Drive 백업: {sorted(p.name for p in (env.persist_root()/'checkpoints').glob('*')) if env.persist_root() and (env.persist_root()/'checkpoints').exists() else '(없음)'}"
    )

# EXP1/EXP2 가 있으면 그게 정답입니다. 없으면 예전 방식으로 이름을 맞춰 봅니다.
ck1, name1 = _find_ckpt(*[c for c in (EXP1, f"s1_{BEST_MODEL}",
                                      f"stage1_resnet50_{STAGE1_CROP}_moderate",
                                      f"stage1_resnet50_{STAGE1_CROP}") if c])
ck2, name2 = _find_ckpt(*[c for c in (EXP2, f"s2_{BEST_MODEL}",
                                      f"stage2_resnet50_{BEST_CROP}_moderate",
                                      f"stage2_resnet50_{BEST_CROP}") if c])
print(f"1단계 체크포인트: {name1}\n2단계 체크포인트: {name2}")

# ★ 백본은 **체크포인트 폴더 이름에서** 읽습니다.
#   예전에는 "03 의 체크포인트는 항상 resnet50" 으로 하드코딩했는데,
#   STEP 6 에서 1단계를 effnetv2_s 로 바꾸면서 깨졌습니다 — effnetv2_s 가중치를
#   resnet50 껍데기에 부으려다 shape mismatch(bn1 24 vs 64)로 죽었습니다.
#   두 단계가 서로 다른 백본을 쓸 수 있으므로 각각 따로 읽습니다.
_k1 = train.model_key_from_exp(name1) or BEST_MODEL
_k2 = train.model_key_from_exp(name2) or BEST_MODEL
if _k1 not in MODEL_BY_KEY or _k2 not in MODEL_BY_KEY:
    raise SystemExit(
        f"❌ 체크포인트 이름에서 모르는 백본이 나왔습니다: 1단계 '{_k1}' / 2단계 '{_k2}'\n"
        f"   MODEL_ZOO 에 있는 키: {sorted(MODEL_BY_KEY)}\n"
        f"   체크포인트 이름: {name1} / {name2}")
spec1, spec2 = MODEL_BY_KEY[_k1], MODEL_BY_KEY[_k2]
print(f"백본 — 1단계 {_k1} / 2단계 {_k2}")
spec = spec2                       # 이후 cfg 는 2단계 기준
cfg = CFG(model_name=spec.timm_name, img_size=IMG_SIZE, exp_name=name2)

m1 = models.load_checkpoint(str(ck1), spec1, len(CLASSES_STAGE1))
m2 = models.load_checkpoint(str(ck2), spec2, len(CLASSES))
print(f"두 단계 모델 로드 완료  (입력 {cfg.img_size}px)")

# ★ 임계값 복원 — 03 의 JSON 이 안 넘어왔을 때만 돕니다.
#   03 과 **같은 정의·같은 fold·같은 모델**이라 같은 값이 나옵니다.
#   (holdout 은 건드리지 않습니다 — 검증셋만 씁니다)
if THR1 is None:
    _cfg1 = CFG(model_name=spec1.timm_name, img_size=IMG_SIZE, exp_name=name1)
    _dl1, _ds1 = data.eval_loader(va1, _cfg1, model=m1, classes=CLASSES_STAGE1)
    # ⚠️ TTA 를 03 과 맞춰야 같은 임계값이 나옵니다 (03 은 cfg1.tta_hflip=True 로 쟀습니다).
    #    안 맞추면 AUROC 가 0.8155 → 0.8143 처럼 미묘하게 달라집니다.
    _, _lg1, _y1 = train.evaluate_loader(m1, _dl1, None, DEV, len(CLASSES_STAGE1),
                                         tta_hflip=CFG().tta_hflip)
    _b1 = evaluate.binary_report(stages.stage1_scores(_lg1),
                                 stages.binary_targets(_y1),
                                 target_recall=CFG().target_recall_stage1)
    THR1 = float(_b1["threshold"])
    print(f"✅ 임계값 재계산: {THR1:.4f}  "
          f"(AUROC {_b1['auroc']:.4f} / precision {_b1['precision_at_target']:.3f})")
    (W/"stage1_threshold.json").write_text(json.dumps({
        "threshold": THR1, "auroc": _b1["auroc"],
        "precision_at_target": _b1["precision_at_target"],
        "target_recall": CFG().target_recall_stage1,
        "stage1_crop": STAGE1_CROP, "stage2_crop": BEST_CROP,
        "stage1_exp": name1, "stage2_exp": name2, "img_size": IMG_SIZE,
        "recovered_by": "notebook 05 (03 의 JSON 이 인계되지 않아 재계산)",
    }, indent=2, ensure_ascii=False))

## 1. 2단계 검증셋 성능 (보정 기준을 잡기 위해)

In [ ]:
dl_va2, ds_va2 = data.eval_loader(va2, cfg, model=m2, classes=CLASSES)
_, logits_va, y_va = train.evaluate_loader(m2, dl_va2, None, DEV, len(CLASSES), tta_hflip=True)
rep_va = evaluate.full_report(logits_va, y_va, CLASSES)
rep_va.plot_confusion()

## 2. 확률 보정 ★

신경망은 **자기 확신이 과합니다.** "95% 확신" 이라고 한 예측 100건 중
실제로는 70건만 맞는 게 흔합니다.

우리는 보호자에게 "신뢰도 72%" 같은 숫자를 보여줄 건데, 그게 거짓말이면 안 되죠.

**온도 스케일링**: logits 를 T 로 나누기만 합니다. 파라미터 딱 1개.
예측 순위는 전혀 안 바뀌니 **정확도는 그대로**, 확률만 정직해집니다.

⚠️ T 는 **검증셋**으로 학습하고 **holdout** 에서 효과를 확인합니다.
holdout 으로 T 를 맞추면 그것도 과적합입니다.

📖 [`docs/basics/08_확률보정과_임계값_결정.md`](../docs/basics/08_확률보정과_임계값_결정.md)

In [ ]:
T = calibrate.fit_temperature(logits_va, y_va)
# ⚠️ 03 에서 바로 온 경우 폴더 이름이 s2_{모델} 이 아닙니다 — 실제 찾은 경로에 씁니다.
(ck2.parent/"temperature.json").write_text(json.dumps({"temperature": T}, indent=2))
train.sync_to_persist(name2, files=("temperature.json",))   # 세션 밖에도 남깁니다

## 3. Holdout 최종 평가 — 파이프라인 기준 ★★

⚠️ **여기서부터는 되돌릴 수 없습니다.**
holdout 결과를 보고 하이퍼파라미터를 고치면 더 이상 holdout 이 아닙니다.
모든 결정이 끝난 뒤 딱 한 번만 여세요.

두 모델을 holdout 전체(정상 포함)에 **같은 순서로** 돌려 이어붙입니다.

In [ ]:
# 각 모델에는 그 모델이 학습한 크롭을 먹입니다
ho1_s2 = crop.switch_tag(ho1, BEST_CROP, verbose=False) if STAGE1_CROP != BEST_CROP else ho1
dl_h1, ds_h1 = data.eval_loader(ho1, cfg, model=m1, classes=CLASSES_STAGE1)
dl_h2, ds_h2 = data.eval_loader(ho1_s2, cfg, model=m2, classes=CLASSES)
assert len(ds_h1.df) == len(ds_h2.df)
assert (ds_h1.df["image_name"].to_numpy() == ds_h2.df["image_name"].to_numpy()).all()

_, lg_h1, _ = train.evaluate_loader(m1, dl_h1, None, DEV, len(CLASSES_STAGE1), tta_hflip=True)
_, lg_h2, _ = train.evaluate_loader(m2, dl_h2, None, DEV, len(CLASSES), tta_hflip=True)

s1_ho = stages.stage1_scores(lg_h1)
y_ho_final = ds_h1.df["label_orig"].to_numpy()

pipe_ho = stages.pipeline_report(s1_ho, lg_h2, y_ho_final, threshold=THR1)
stages.plot_pipeline_confusion(pipe_ho)

In [ ]:
# 2단계만 따로 본 holdout 점수 (파이프라인과 비교하기 위해)
mask_lesion = y_ho_final != NORMAL_LABEL
y_ho2 = torch.tensor([CLASSES.index(c) for c in y_ho_final[mask_lesion]])
rep_ho2 = evaluate.full_report(lg_h2[torch.as_tensor(mask_lesion)], y_ho2, CLASSES)
print(f"\n2단계만: macro-F1 {rep_ho2.metrics['macro_f1']:.4f}")
print(f"파이프라인: 최종 macro-F1 {pipe_ho['final_macro_f1']:.4f}, "
      f"스크리닝 recall {pipe_ho['lesion_screening_recall']:.4f}")
print("💡 두 숫자의 격차가 '1단계가 깎아먹는 양' 입니다. 보고할 때는 파이프라인 숫자를 쓰세요.")

In [ ]:
cal = calibrate.report(logits_va, y_va, lg_h2[torch.as_tensor(mask_lesion)], y_ho2)
probs_after = calibrate.apply(lg_h2[torch.as_tensor(mask_lesion)], T)
calibrate.reliability_diagram(evaluate.softmax_np(lg_h2[torch.as_tensor(mask_lesion)]),
                              probs_after, y_ho2.numpy())

## 4. Grad-CAM — 필수 검증 게이트 ★★

**정확도가 아무리 좋아도 여기서 통과 못 하면 그 모델은 실패입니다.**

이 데이터는 병변이 이미지의 5% 미만이고 배경이 제각각입니다.
모델이 병변이 아니라 진료대 무늬, 조명, 털 색을 보고 맞힐 수 있고,
그 단서가 클래스와 상관이 있으면 **검증 점수까지 잘 나옵니다.**

숫자로는 절대 못 잡습니다. 그림을 봐야 합니다.

In [ ]:
explain.grid(m2, va2, cfg, n=8)

In [ ]:
# 틀린 예측만 골라 보기 — 어디를 보고 틀렸는지가 개선의 힌트
va2b = va2.copy()
va2b["pred"] = [CLASSES[i] for i in evaluate.softmax_np(logits_va).argmax(1)]
explain.grid(m2, va2b, cfg, n=8, only_correct=False)

In [ ]:
# 수치화: CAM 이 실제 병변 박스와 얼마나 겹치는가
# 원본 이미지가 없는 환경(크롭만 업로드)이면 자동으로 크롭 좌표계로 계산합니다
overlap = explain.lesion_overlap_score(m2, va2, cfg, n=150, frame="auto")

### 🚦 게이트 판정

- `median_lift` ≥ 1.3 → 병변을 보고 있음, 통과
- `median_lift` < 1.3 → **배경 학습 의심**. 정확도와 무관하게 재작업

재작업 방향: 크롭 margin 축소 / 배경 증강 강화 / 세그멘테이션 마스킹

In [ ]:
if not overlap:
    print("⚠️ 정렬도를 계산하지 못했습니다 — 게이트를 판정할 수 없습니다.")
    print("   bbox 가 있는 행이 있는지, crop_path/img_w/img_h 가 살아있는지 확인하세요.")
    print("   이 상태로 배포 판단을 하면 안 됩니다.")
else:
    assert overlap["median_lift"] >= 1.3, (
        f"CAM-병변 정렬도 {overlap['median_lift']:.2f} < 1.3 — 배경을 보고 있을 가능성이 큽니다.\n"
        "정확도와 무관하게 재작업 대상입니다. docs/cautions/03 참고."
    )
    print(f"✅ Grad-CAM 게이트 통과 — median_lift {overlap['median_lift']:.2f} "
          f"(프레임: {overlap['frame']})")

### 🚦 실사용 견고성 게이트 ★

Grad-CAM 이 "병변을 보고 있다"고 해도, 그게 **어떤 배율에서만** 통하는 것일 수 있습니다.
보호자 사진의 배율과 병변 위치는 우리가 통제할 수 없습니다.

holdout 을 일부러 그렇게 망가뜨려 하락폭을 잽니다. 배포 판단의 마지막 관문입니다.

In [ ]:
from src import robust

rb = robust.report(m2, ho2, cfg, CLASSES, n=2000)
scale_drop = rb["scale"].get("_summary", {}).get("rel_drop", float("nan"))
shift_drop = rb["shift"].get("_summary", {}).get("rel_drop", float("nan"))

In [ ]:
if scale_drop == scale_drop and scale_drop > 0.30:
    print(f"🚨 배율 하락 {scale_drop:.0%} — 배포하면 안 됩니다.")
    print("   모델이 크롭 배율에 의존하고 있습니다. 보호자 사진에는 그 배율이 없습니다.")
    print("   → 고정 픽셀 크롭(f320) 또는 강한 배율 증강으로 다시 학습하세요 (노트북 03).")
elif scale_drop == scale_drop and scale_drop > 0.15:
    print(f"⚠️ 배율 하락 {scale_drop:.0%} — 실사용 성능은 holdout 점수보다 낮을 겁니다.")
    print("   배포한다면 그 사실을 문서에 명시하세요.")
else:
    print(f"✅ 배율 하락 {scale_drop:.0%}")

if shift_drop == shift_drop and shift_drop > 0.30:
    print(f"🚨 위치 하락 {shift_drop:.0%} — 병변이 화면 가운데 있을 때만 동작합니다.")
    print("   → 사용자에게 '병변을 가운데 두고 찍어주세요' 를 안내하거나, 위치 증강을 넣으세요.")
else:
    print(f"✅ 위치 하락 {shift_drop:.0%}")

## 5. 임계값과 거절(abstention) 설계

두 가지 임계값이 있습니다. 헷갈리기 쉬우니 구분하세요:

| 임계값 | 무엇을 정하나 | 기준 |
|---|---|---|
| **1단계 임계값** (`THR1`) | 병원에 가보라고 알릴지 | 재현율 ≥ 0.95 — 놓치지 않기 |
| **거절 임계값** (`abstain`) | 병변 **종류**를 말할지 | 틀릴 위험 ≤ 20% |

거절되면 종류를 말하지 않고 "판단이 어려운 사진입니다" 로 물러섭니다.
단, **1단계에서 이상이라고 판단했으면 거절해도 병원 안내는 유지**합니다 —
종류를 모른다는 게 괜찮다는 뜻은 아니니까요.

In [ ]:
cr = calibrate.coverage_risk_curve(probs_after, y_ho2.numpy())
thr_abstain = calibrate.suggest_abstain_threshold(probs_after, y_ho2.numpy(), max_risk=0.20)

### 1단계 임계값을 holdout 에서 재확인

⚠️ 임계값은 **검증셋에서 정하고** holdout 에서는 확인만 합니다.
holdout 점수가 목표(0.95)에 못 미치면 임계값을 고치는 게 아니라
"검증셋 기준으로 정한 임계값이 새 데이터에서는 recall 0.93 이었다"고 **보고**합니다.

In [ ]:
ybin_ho = (y_ho_final != NORMAL_LABEL).astype(int)
print(f"검증셋에서 정한 임계값 {THR1:.4f} 를 holdout 에 적용:")
print(f"  스크리닝 recall  {pipe_ho['lesion_screening_recall']:.4f}   (목표 ≥ 0.95)")
print(f"  헛알림 비율      {pipe_ho['false_alarm_rate']:.4f}")

# 참고용: holdout 에서 0.95를 만족하려면 어디였어야 하는가 (보고용, 채택하지 마세요)
ref = evaluate.binary_report(s1_ho, ybin_ho, target_recall=0.95)
print(f"\n(참고) holdout 기준 최적 임계값은 {ref['threshold']:.4f} 였습니다 — "
      "이 값을 채택하면 holdout 이 오염됩니다.")
print(f"두 값의 차이 {abs(ref['threshold'] - THR1):.4f} 가 크면 검증셋이 작거나 분포가 다른 것입니다.")

---
## 5-b. 촬영 가이드 (capture guideline) 도출 ★

배율 강건성(scale robustness)은 **모델링으로 못 잡았습니다.** 해상도로 8~9%p 를
줄인 뒤(224→384), 증강(augmentation) 7종을 한 실행에서 비교했지만 전부 잡음 안이었고
`zoom_both` 는 오히려 4%p 악화시켰습니다.
→ [`docs/results/STEP4B_증강스윕_실측.md`](../docs/results/STEP4B_증강스윕_실측.md)

남은 길은 **애초에 나쁜 배율이 안 들어오게 입력을 제한**하는 것입니다
(멘토 피드백 2번: "정확도가 가장 높은 scale 로 촬영하도록 가이드").

그러려면 "얼마나 가까이" 를 **숫자로** 말할 수 있어야 합니다. 그 숫자를 여기서 뽑습니다.

### 학습은 안 합니다

이미 학습된 모델에 **배율만 바꿔 추론**할 뿐입니다. 몇 분이면 끝납니다.

### 왜 격자를 촘촘하게 하나

지금까지 쓰던 5개 점(0.5 / 0.71 / 1 / 1.41 / 2)은 간격이 √2 라 너무 성깁니다.
03 실측으로 계산해보면 **밴드가 한 점으로 무너집니다**:

| 허용 하락 | 5개 점으로 계산한 밴드 |
|---|---|
| 5% 이내 | 1.0x ~ 1.0x ← 쓸 수 없음 |
| 10% 이내 | 1.0x ~ 1.41x |

1.41x 가 −5.2%, 0.71x 가 −10.4% 로 **둘 다 기준을 아슬하게 놓치기** 때문입니다.
0.85x / 1.2x 를 재봐야 실제 경계가 나옵니다.

### 배율을 "화면 점유율" 로 바꿉니다

보호자는 "1.2배" 를 모르지만 **"화면 절반"** 은 압니다.
학습 크롭이 `m1.5` 니 1x 에서 병변이 화면 가로의 1/1.5 = **67%** 를 차지합니다.


In [ ]:
# 배율 축 — 촘촘한 격자로 다시 잽니다 (추론만, 학습 없음)
guide_scale = robust.usable_range(
    m2, va2, cfg, CLASSES,
    zooms=(0.5, 0.6, 0.7, 0.85, 1.0, 1.2, 1.4, 1.7, 2.0),
    tolerances=(0.05, 0.10),
    crop_margin=CROP_MARGIN,  # 학습 크롭에서 유도 (m1.5→67% / m2.5→40%)
    n=3000, device=DEV)

# 위치 축 — 중앙에서 얼마나 벗어나도 되는지
guide_shift = robust.usable_shift(
    m2, va2, cfg, CLASSES,
    fracs=(0.0, 0.05, 0.10, 0.15, 0.20, 0.30),
    tolerance=0.05, n=3000, device=DEV)


In [ ]:
# ★ 보호자에게 보여줄 문구로 바꿉니다
b5 = guide_scale["bands"].get(0.05)
b10 = guide_scale["bands"].get(0.10)
occ = guide_scale["occupancy"]
mx = guide_shift.get("max_shift")

print("=" * 62)
print(" 촬영 가이드 (이 블록을 앱 UI 문구로 쓰세요)")
print("=" * 62)
if b5:
    print(f"  권장  : 병변이 화면 가로의 {occ[b5[0]]:.0%} ~ {occ[b5[1]]:.0%} 를 채우도록")
    print(f"          (배율 {b5[0]}x ~ {b5[1]}x · 성능 하락 5% 이내)")
if b10:
    print(f"  허용  : {occ[b10[0]]:.0%} ~ {occ[b10[1]]:.0%}  "
          f"(배율 {b10[0]}x ~ {b10[1]}x · 하락 10% 이내)")
if mx is not None:
    print(f"  위치  : 병변이 화면 중앙에서 {mx:.0%} 이내에 있도록")
print()
print("  → 촬영 UI 에 가이드 프레임을 띄우고, 벗어나면 셔터를 막거나")
print("     '조금 더 가까이 찍어주세요' 를 띄우는 방식으로 구현합니다.")
print("  → 이 구간을 벗어난 사진은 추론 대신 **다시 찍어달라고** 하는 게 맞습니다")
print("     (5번의 거절(abstention) 임계값과 같은 목적).")
print("=" * 62)

# 리포트에 남깁니다
import json
_W = env.work_root(); (_W/"reports").mkdir(parents=True, exist_ok=True)
(_W/"reports"/"capture_guide.json").write_text(json.dumps({
    "scale": {"peak": guide_scale["peak"], "table": guide_scale["table"],
              "bands": {str(k): v for k, v in guide_scale["bands"].items()},
              "occupancy": {str(k): v for k, v in guide_scale["occupancy"].items()},
              "crop_margin": guide_scale["crop_margin"]},
    "shift": {"max_shift": mx, "table": guide_shift["table"]},
}, indent=2, ensure_ascii=False))
print(f"저장: {_W/'reports'/'capture_guide.json'}")


## 6. 실제 사용 시뮬레이션

사용자가 사진 한 장을 올렸을 때 무엇이 보이는지 확인합니다.
`TwoStageEngine` 이 1단계 → 2단계를 실제 서비스와 같은 순서로 돌립니다.

⚠️ **병변 이름은 나오지 않습니다.** 2026-08-26 에 출력 형식을 바꿨습니다 —
1단계가 '이상' 이라고 하면 2단계 확률을 **여섯 개 전부** 보여주고 어느 쪽인지는
판단할 수 없다고 말한 뒤 진료를 권합니다. holdout 에서 그 이름이 **56.6%**
틀렸기 때문입니다 (`docs/cautions/03_의료AI_안전설계_원칙.md` §7-B).

In [ ]:
cfg_s2 = CFG(**{**cfg.to_dict(), "abstain_threshold": thr_abstain})
eng1 = infer.Engine(m1, cfg, CLASSES_STAGE1, temperature=1.0)
eng2 = infer.Engine(m2, cfg_s2, CLASSES, temperature=T)
pipeline = infer.TwoStageEngine(eng1, eng2, threshold=THR1)

# 정상 1장 + 병변 2장을 골라 봅니다
sample = pd.concat([
    ho1[ho1["label_orig"] == NORMAL_LABEL].sample(1, random_state=0),
    ho1[ho1["label_orig"] != NORMAL_LABEL].sample(2, random_state=0),
]) if (ho1["label_orig"] == NORMAL_LABEL).any() else ho1.sample(3, random_state=0)

for _, r in sample.iterrows():
    print("=" * 62)
    print("정답:", r["label_orig"], CLASS_KO.get(r["label_orig"], ""))
    pipeline.show(r["crop_path"])
    print()

## 7. 최종 리포트 저장

In [ ]:
summary = {
    "model": BEST_MODEL, "stage2_crop": BEST_CROP, "stage1_crop": STAGE1_CROP,
    "stage1": {"threshold": THR1,
               "holdout_screening_recall": pipe_ho["lesion_screening_recall"],
               "holdout_false_alarm_rate": pipe_ho["false_alarm_rate"],
               "holdout_lesion_missed": pipe_ho["lesion_missed"]},
    "stage2_only": {"val_macro_f1": rep_va.metrics["macro_f1"],
                    "val_ci": list(rep_va.ci[1:]),
                    "holdout_macro_f1": rep_ho2.metrics["macro_f1"],
                    "holdout_ci": list(rep_ho2.ci[1:]),
                    "holdout_per_class_recall": rep_ho2.metrics["per_class"]["recall"]},
    "pipeline_holdout": {k: v for k, v in pipe_ho.items()
                         if k not in ("confusion", "per_class")},
    "pipeline_per_class": pipe_ho["per_class"],
    "calibration": cal,
    "temperature": T,
    "cam_lesion_overlap": overlap,
    "robustness": {"scale_rel_drop": scale_drop, "shift_rel_drop": shift_drop},
    "abstain_threshold": thr_abstain,
    "coverage_risk": cr,
}
p = W/"reports"/f"final_{BEST_MODEL}.json"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print("저장:", p)
print(json.dumps(summary["pipeline_holdout"], indent=2, ensure_ascii=False))

---
## ✅ 배포 전 체크리스트

**파이프라인 기준**으로 확인하세요. 2단계만 좋아도 소용없습니다.

- [ ] **스크리닝 recall ≥ 0.95** (holdout) ← 가장 중요
- [ ] holdout 성능이 검증셋과 크게 다르지 않다 (차이 크면 과적합)
- [ ] 모든 병변 클래스의 recall 이 0.5 이상이다
- [ ] A5·A6(위험 병변)의 recall 이 특히 낮지 않다
- [ ] 보정 후 ECE < 0.10
- [ ] Grad-CAM 이 병변을 보고 있다 (median_lift ≥ 1.3)
- [ ] **배율 교란 하락 < 15%** — 보호자가 다른 거리에서 찍어도 버팀
- [ ] **위치 교란 하락 < 15%** — 병변이 정중앙이 아니어도 버팀
- [ ] 거절 임계값이 정해져 있다
- [ ] 모든 출력에 "진단이 아님" 문구가 붙는다
- [ ] **크롭 전제를 확인했다** — `m1.5`/`m2.5` 로 학습했다면 실제 사용자 사진에는
      병변 박스가 없습니다. `full` 점수를 기대치로 쓰거나 검출 모델을 앞에 붙이세요.

📖 반드시 읽기:
- [`docs/cautions/03_의료AI_안전설계_원칙.md`](../docs/cautions/03_의료AI_안전설계_원칙.md)
- [`docs/cautions/08_2단계_파이프라인_설계_주의점.md`](../docs/cautions/08_2단계_파이프라인_설계_주의점.md)